# N-Gram + BPC Final Notebook

This notebook is intended to generate a **final-ready** export for the report.

It runs:

- `word`, `char`, `bpe`
- `1-gram`, `2-gram`, `3-gram`
- BPC as the main cross-tokenizer metric
- PPL as a complementary metric
- qualitative prediction and sentence-scoring examples

Compared with earlier versions, this notebook now uses **character-based split limits** so each tokenizer is evaluated on the same amount of original text.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
from typing import Optional

IS_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/HatakekkSheeshh/text-preprocess-tokenization.git"
GIT_REF = "bpc-metric"
REPO_NAME = "text-preprocess-tokenization"
REPO_DIR = None  # e.g. "/root/text-preprocess-tokenization" if the repo already exists
AUTO_CLONE_IF_MISSING = True
AUTO_PULL_LATEST = False


def looks_like_repo_root(path: Path) -> bool:
    return (path / "requirements.txt").exists() and (path / "src").exists() and (path / "main.py").exists()


def find_repo_root(start: Path) -> Optional[Path]:
    common_roots = [start, *start.parents, Path.home(), Path("/root"), Path("/content"), Path("/workspace"), Path("/mnt"), Path("/tmp")]
    seen = set()
    candidates = []
    for root in common_roots:
        if not root.exists():
            continue
        candidates.append(root)
        candidates.append(root / REPO_NAME)
        try:
            for child in root.iterdir():
                if child.is_dir() and child.name == REPO_NAME:
                    candidates.append(child)
        except OSError:
            pass

    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if looks_like_repo_root(candidate):
            return candidate
    return None


def run_git(*args: str) -> None:
    subprocess.run(["git", *args], check=True)


if REPO_DIR is not None:
    project_root = Path(REPO_DIR).expanduser().resolve()
    if not looks_like_repo_root(project_root):
        raise FileNotFoundError(f"REPO_DIR does not look like the repo root: {project_root}")
else:
    detected_root = find_repo_root(Path.cwd().resolve())
    if detected_root is None and AUTO_CLONE_IF_MISSING:
        clone_parent = Path("/content") if Path("/content").exists() else Path.home()
        project_root = (clone_parent / REPO_NAME).resolve()
        if not project_root.exists():
            run_git("clone", REPO_URL, str(project_root))
        run_git("-C", str(project_root), "checkout", GIT_REF)
    elif detected_root is None:
        raise FileNotFoundError(
            "Could not find the project root automatically. "
            "Set REPO_DIR to the repo path, or allow AUTO_CLONE_IF_MISSING."
        )
    else:
        project_root = detected_root

if AUTO_PULL_LATEST:
    run_git("-C", str(project_root), "fetch", "origin")
    run_git("-C", str(project_root), "checkout", GIT_REF)
    run_git("-C", str(project_root), "pull", "--ff-only", "origin", GIT_REF)

os.chdir(project_root)
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python import root added: {PROJECT_ROOT}")
print(f"Using branch/config target: {GIT_REF}")

In [ ]:
!pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"

In [ ]:
import json
import shutil

import pandas as pd

from src.datasets.load_data import load
from src.training.train_ngram import NGramTrainingConfig, train_ngram_language_model

METRICS_ROOT = PROJECT_ROOT / "outputs" / "metrics" / "ngram"
ARTIFACT_ROOT = PROJECT_ROOT / "outputs" / "artifacts" / "ngram"
REPORT_ROOT = PROJECT_ROOT / "outputs" / "report_tables" / "ngram"


def load_metrics(run_name: str) -> dict:
    return json.loads((METRICS_ROOT / f"{run_name}.json").read_text(encoding="utf-8"))


def make_run_name(dataset_name: str, tokenizer_name: str, order: int, profile: str) -> str:
    return f"report_{dataset_name.replace('-', '_')}_{tokenizer_name}_{order}gram_{profile}_bpc"


def build_comparison_row(metrics: dict) -> dict:
    train_split = metrics["splits"]["train"]
    validation_split = metrics["splits"]["validation"]
    test_split = metrics["splits"]["test"]
    return {
        "dataset": metrics["config"]["dataset_name"],
        "tokenizer": metrics["tokenizer"]["type"],
        "n_gram": f"{metrics['model']['order']}-gram",
        "order": metrics["model"]["order"],
        "vocab_size": metrics["tokenizer"]["vocab_size"],
        "train_tokens": train_split["num_tokens"],
        "train_characters": train_split["num_characters"],
        "validation_characters": validation_split["num_characters"],
        "test_characters": test_split["num_characters"],
        "tokenizer_fit_s": round(metrics["timing"]["tokenizer_fit_seconds"], 4),
        "model_fit_s": round(metrics["timing"]["model_fit_seconds"], 4),
        "total_s": round(metrics["timing"]["total_seconds"], 4),
        "val_bpc": round(validation_split["bits_per_character"], 4),
        "test_bpc": round(test_split["bits_per_character"], 4),
        "val_avg_nll": round(validation_split["average_negative_log_likelihood"], 4),
        "test_avg_nll": round(test_split["average_negative_log_likelihood"], 4),
        "val_ppl": round(validation_split["perplexity"], 4),
        "test_ppl": round(test_split["perplexity"], 4),
        "run_name": metrics["run_name"],
    }


def build_prediction_rows(metrics: dict) -> list[dict]:
    rows = []
    for item in metrics["prediction_contexts"]:
        top_predictions = ", ".join(
            f"{pred['token']} ({pred['probability']:.4g})" for pred in item["predictions"]
        )
        rows.append(
            {
                "dataset": metrics["config"]["dataset_name"],
                "tokenizer": metrics["tokenizer"]["type"],
                "n_gram": f"{metrics['model']['order']}-gram",
                "context": item["context_text"],
                "top_predictions": top_predictions,
            }
        )
    return rows


def build_scored_text_rows(metrics: dict) -> list[dict]:
    rows = []
    for item in metrics["scored_texts"]:
        rows.append(
            {
                "dataset": metrics["config"]["dataset_name"],
                "tokenizer": metrics["tokenizer"]["type"],
                "n_gram": f"{metrics['model']['order']}-gram",
                "text": item["text"],
                "avg_nll": round(item["average_negative_log_likelihood"], 4),
                "bpc": round(item["bits_per_character"], 4),
                "ppl": round(item["perplexity"], 4),
            }
        )
    return rows


## Experiment configuration

Recommended defaults:

- `PROFILE = "medium"` for report-oriented runs
- `TOKENIZER_NAMES = ["word", "char", "bpe"]`
- `NGRAM_ORDERS = [1, 2, 3]`

This notebook uses **character-based split limits** in the experiment profiles below, so each tokenizer sees the same amount of original text per split.

In [ ]:
DATASET_NAME = "text8"
PROFILE = "medium"  # quick | medium | full

TOKENIZER_NAMES = ["word", "char", "bpe"]
NGRAM_ORDERS = [1, 2, 3]
LAPLACE_ALPHA = 1.0
TOP_K = 5

MAX_VOCAB_SIZE_BY_TOKENIZER = {
    "word": 50_000,
    "char": None,
    "bpe": 16_000,
}

PREDICTION_CONTEXTS = [
    "the history",
    "in the",
]

SCORE_TEXTS = [
    "the history of science",
    "science of history the",
]

PROFILES = {
    "quick": {
        "text8": {
            "max_fit_texts": None,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
        "enwik8": {
            "max_fit_texts": 1,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
        "wikitext-103": {
            "max_fit_texts": 500,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
        "one-billion-word": {
            "max_fit_texts": 500,
            "max_fit_characters": 200_000,
            "max_train_tokens": None,
            "max_train_characters": 200_000,
            "max_validation_tokens": None,
            "max_validation_characters": 50_000,
            "max_test_tokens": None,
            "max_test_characters": 50_000,
        },
    },
    "medium": {
        "text8": {
            "max_fit_texts": None,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
        "enwik8": {
            "max_fit_texts": 1,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
        "wikitext-103": {
            "max_fit_texts": 2_000,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
        "one-billion-word": {
            "max_fit_texts": 2_000,
            "max_fit_characters": 1_000_000,
            "max_train_tokens": None,
            "max_train_characters": 1_000_000,
            "max_validation_tokens": None,
            "max_validation_characters": 250_000,
            "max_test_tokens": None,
            "max_test_characters": 250_000,
        },
    },
    "full": {
        "text8": {
            "max_fit_texts": None,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
        "enwik8": {
            "max_fit_texts": 1,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
        "wikitext-103": {
            "max_fit_texts": None,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
        "one-billion-word": {
            "max_fit_texts": None,
            "max_fit_characters": None,
            "max_train_tokens": None,
            "max_train_characters": None,
            "max_validation_tokens": None,
            "max_validation_characters": None,
            "max_test_tokens": None,
            "max_test_characters": None,
        },
    },
}

limits = PROFILES[PROFILE][DATASET_NAME].copy()
print({
    "dataset": DATASET_NAME,
    "profile": PROFILE,
    "tokenizers": TOKENIZER_NAMES,
    "orders": NGRAM_ORDERS,
    **limits,
})

In [ ]:
load(DATASET_NAME)
print(f"Dataset ready: {DATASET_NAME}")

In [ ]:
comparison_rows = []
prediction_rows = []
scored_text_rows = []
run_names = []

for tokenizer_name in TOKENIZER_NAMES:
    for order in NGRAM_ORDERS:
        run_name = make_run_name(DATASET_NAME, tokenizer_name, order, PROFILE)
        print(f"Running {run_name} ...")

        config = NGramTrainingConfig(
            dataset_name=DATASET_NAME,
            tokenizer_name=tokenizer_name,
            order=order,
            alpha=LAPLACE_ALPHA,
            max_vocab_size=MAX_VOCAB_SIZE_BY_TOKENIZER[tokenizer_name],
            max_fit_texts=limits["max_fit_texts"],
            max_fit_characters=limits["max_fit_characters"],
            max_train_tokens=limits["max_train_tokens"],
            max_train_characters=limits["max_train_characters"],
            max_validation_tokens=limits["max_validation_tokens"],
            max_validation_characters=limits["max_validation_characters"],
            max_test_tokens=limits["max_test_tokens"],
            max_test_characters=limits["max_test_characters"],
            run_name=run_name,
        )

        summary = train_ngram_language_model(
            config,
            prediction_contexts=PREDICTION_CONTEXTS,
            score_texts=SCORE_TEXTS,
            top_k=TOP_K,
        )

        metrics = load_metrics(summary["run_name"])
        run_names.append(summary["run_name"])
        comparison_rows.append(build_comparison_row(metrics))
        prediction_rows.extend(build_prediction_rows(metrics))
        scored_text_rows.extend(build_scored_text_rows(metrics))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["tokenizer"] = pd.Categorical(comparison_df["tokenizer"], categories=TOKENIZER_NAMES, ordered=True)
comparison_df = comparison_df.sort_values(["tokenizer", "order"]).reset_index(drop=True)

report_df = comparison_df[[
    "tokenizer",
    "n_gram",
    "train_tokens",
    "train_characters",
    "tokenizer_fit_s",
    "model_fit_s",
    "val_bpc",
    "val_avg_nll",
    "val_ppl",
    "test_bpc",
    "test_avg_nll",
    "test_ppl",
]]

display(report_df)

In [ ]:
prediction_df = pd.DataFrame(prediction_rows)
prediction_df["tokenizer"] = pd.Categorical(prediction_df["tokenizer"], categories=TOKENIZER_NAMES, ordered=True)
prediction_df = prediction_df.sort_values(["tokenizer", "n_gram", "context"]).reset_index(drop=True)

scored_text_df = pd.DataFrame(scored_text_rows)
scored_text_df["tokenizer"] = pd.Categorical(scored_text_df["tokenizer"], categories=TOKENIZER_NAMES, ordered=True)
scored_text_df = scored_text_df.sort_values(["tokenizer", "n_gram", "text"]).reset_index(drop=True)

print("Prediction examples")
display(prediction_df)

print("Sentence scoring examples")
display(scored_text_df)

In [ ]:
export_dir = REPORT_ROOT / DATASET_NAME / PROFILE
zip_base = PROJECT_ROOT / f"report_export_{DATASET_NAME.replace('-', '_')}_{PROFILE}_ngram_bpc"

if export_dir.exists():
    shutil.rmtree(export_dir)

(export_dir / "metrics").mkdir(parents=True, exist_ok=True)
(export_dir / "artifacts").mkdir(parents=True, exist_ok=True)

report_df.to_csv(export_dir / "quantitative_comparison.csv", index=False)
prediction_df.to_csv(export_dir / "prediction_examples.csv", index=False)
scored_text_df.to_csv(export_dir / "sentence_scoring_examples.csv", index=False)

(export_dir / "run_names.json").write_text(json.dumps(run_names, indent=2), encoding="utf-8")
(export_dir / "config.json").write_text(
    json.dumps(
        {
            "dataset": DATASET_NAME,
            "profile": PROFILE,
            "tokenizers": TOKENIZER_NAMES,
            "orders": NGRAM_ORDERS,
            "limits": limits,
            "max_vocab_size_by_tokenizer": MAX_VOCAB_SIZE_BY_TOKENIZER,
            "prediction_contexts": PREDICTION_CONTEXTS,
            "score_texts": SCORE_TEXTS,
        },
        indent=2,
    ),
    encoding="utf-8",
)

for run_name in run_names:
    shutil.copy2(METRICS_ROOT / f"{run_name}.json", export_dir / "metrics" / f"{run_name}.json")
    shutil.copytree(ARTIFACT_ROOT / run_name, export_dir / "artifacts" / run_name, dirs_exist_ok=True)

zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=export_dir)
print("Export directory:", export_dir)
print("Created zip:", zip_path)

In [ ]:
if IS_COLAB:
    from google.colab import files
    files.download(str(Path(f"{zip_base}.zip")))
else:
    print(Path(f"{zip_base}.zip"))